# 睡眠剥夺 scRNA-seq: pySCENIC GRN 推断 + CellOracle 虚拟敲除\n## 在 Google Colab 中运行 (免费 GPU/12GB+ RAM)\n运行前：菜单栏 → 修改 → 笔记本设置 → 选择 GPU (T4) 运行时

In [ ]:
# ============ 第1步：安装依赖 ============\n!pip install scanpy pandas numpy matplotlib seaborn scipy leidenalg -q\n!pip install pyscenic celloracle -q\n!pip install anndata h5py -q\n\n# 注意: pySCENIC 需要配套的 motif 数据库\n# 这些文件较大，Colab 中用 wget 下载

In [ ]:
# ============ 第2步：下载 pySCENIC 参考数据库 ============\n# 小鼠参考文件 (cisTarget motif 数据库)\n!mkdir -p ./scenic_data\n!wget -O ./scenic_data/mm10__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather \\\n  https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl -q 2>/dev/null || \\\n  echo "motif 文件需从 https://resources.aertslab.org/cistarget/ 手动下载"\n\n# 更可靠的下载方式：\n# 1. Motif ranking: https://resources.aertslab.org/cistarget/motif2tf/\n# 2. TF list: https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl\n# 3. Motif annotation: https://resources.aertslab.org/cistarget/motif2tf/motifs-v9-nr.mgi-m0.001-o0.0.tbl

In [ ]:
# ============ 第3步：上传你的 Seurat 数据 ============\n# 把本地 Seurat 对象转为 h5ad 格式上传\n\n# 本地 R 中执行这段代码导出数据：\n# library(SeuratDisk)\n# SaveH5Seurat(obj, "sleep_sd.h5Seurat")\n# Convert("sleep_sd.h5Seurat", dest = "h5ad")\n\n# 在 Colab 中上传 h5ad 文件\nfrom google.colab import files\n# uploaded = files.upload()\n\nimport scanpy as sc\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nprint(f'scanpy version: {sc.__version__}')\nsc.settings.set_figure_params(dpi=100, facecolor='white')

In [ ]:
# ============ 第4步：加载数据 ============\n# adata = sc.read_h5ad('sleep_sd.h5ad')\n\n# 如果是直接从 GEO 下载的 10x 格式：\n# adata = sc.read_10x_mtx('filtered_feature_bc_matrix/')\n\n# 做基础预处理\n# adata.var_names_make_unique()\n# sc.pp.filter_cells(adata, min_genes=200)\n# sc.pp.filter_genes(adata, min_cells=3)\n# sc.pp.normalize_total(adata, target_sum=1e4)\n# sc.pp.log1p(adata)\n\nprint('数据加载完成，等待读取 h5ad 文件')\n# print(adata)

In [ ]:
# ============ 第5步：pySCENIC GRN 推断 ============\nimport pyscenic\nfrom pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase\nfrom pyscenic.cli.utils import load_signatures\n\n# 加载 motif ranking 数据库\nrankings_db = './scenic_data/mm10__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather'\n\n# 方法1: 从 Seurat 导出的 h5ad 提取表达矩阵\n# expr_mat = adata.to_df().T  # gene x cell\n\n# 方法2: 从 CSV 读取（如果你从 R 导出了表达矩阵）\n# expr_mat = pd.read_csv('expression_matrix.csv', index_col=0)\n\nprint('请确保已上传表达矩阵文件')\n\n# pySCENIC 三步走：\n# Step 1: GRNBoost2 - 推断共表达模块\n# !pyscenic grn expression_matrix.csv \\\n#     mm_mgi_tfs.txt \\\n#     -o adjacencies.csv \\\n#     --num_workers 4\n\n# Step 2: RcisTarget - 顺式调控模块分析\n# !pyscenic ctx adjacencies.csv \\\n#     rankings_db \\\n#     --annotations_fname motifs-tbl.txt \\\n#     --expression_mtx_fname expression_matrix.csv \\\n#     --output regulons.csv \\\n#     --num_workers 4\n\n# Step 3: AUCell - 计算每个细胞的 TF 活性\n# !pyscenic aucell expression_matrix.csv \\\n#     regulons.csv \\\n#     --output aucell.csv \\\n#     --num_workers 4\n\n# 简化方法（在 Python 中直接调用）：\nfrom arboreto.algo import grnboost2\n\nprint('pySCENIC 流程准备就绪，需要表达矩阵和 TF 列表文件。')

In [ ]:
# ============ 第6步：CellOracle 虚拟敲除 ============\nimport celloracle as co\nprint(f'celloracle version: {co.__version__}')\n\n# 6.1 初始化 Oracle 对象\n# oracle = co.Oracle()\n\n# 6.2 加载数据（从 scRNA-seq）\n# oracle.import_anndata_as_raw_count(\n#     adata=adata,\n#     cluster_column_name='cell_type',\n#     embedding_name='X_umap'\n# )\n\n# 6.3 用 pySCENIC 结果构建 GRN 模型\n# oracle.import_TF_data(TF_info_matrix=links)  # from pySCENIC\n# oracle.fit_GRN_for_simulation()\n\n# 6.4 设定虚拟敲除目标 TF\n# target_TF = 'Nr3c1'  # 糖皮质激素受体，睡眠应激关键 TF\n# oracle.simulate_shift(\n#     perturb_condition={target_TF: 0.0},  # 将 TF 表达敲低到 0\n#     n_propagation=3\n# )\n\n# 6.5 计算敲除前后的向量场变化\n# oracle.estimate_transition_prob(n_neighbors=200, knn_random=True)\n# oracle.calculate_embedding_shift()\n\nprint('CellOracle 虚拟敲除流程准备就绪。')\nprint('具体参数需根据你的数据和目标 TF 调整。')

In [ ]:
# ============ 第7步：可视化虚拟敲除结果 ============\n\n# 7.1 扰动向量场图\n# oracle.plot_quiver(scale=30)  # 箭头显示预测的变化方向\n\n# 7.2 过渡概率热图\n# co.plot.plot_transition_prob(oracle)\n\n# 7.3 预测的差异表达基因\n# predicted_de = oracle.delta_embedding  # 敲除后的转录变化\n\n# 7.4 自定义可视化：预测受影响的基因\ndef plot_perturbation_effect(oracle, target_tf, top_n_genes=20):\n    \n    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n    \n    # 左侧：向量场箭头\n    ax = axes[0]\n    # oracle.plot_quiver(ax=ax, scale=30)\n    ax.set_title(f'{target_tf} Knockout: Predicted Cell Fate Shift')\n    \n    # 右侧：预测受影响的下游基因 bar plot\n    ax = axes[1]\n    # n_genes = predicted_de.nlargest(top_n_genes, 'abs_change')\n    # sns.barplot(data=n_genes, x='change', y='gene', ax=ax)\n    ax.set_title(f'Top {top_n_genes} Downstream Genes Affected')\n    ax.axvline(0, color='grey', linestyle='--')\n    \n    plt.tight_layout()\n    return fig\n\n# plot_perturbation_effect(oracle, 'Nr3c1')\n\nprint('可视化代码就绪。')

## 睡眠剥夺中可关注的转录因子\n\n基于文献，以下是睡眠剥夺研究中已知的关键 TF：\n\n| TF | 功能 | 文献支持 |\n|-----|------|----------|\n| **Nr3c1** (GR) | 糖皮质激素受体，应激核心调控者 | 高 |\n| **Crem** | cAMP 响应元件，节律调控 | 高 |\n| **Per1/Per2** | 昼夜节律核心 | 高 |\n| **Fos/Jun** | 即早基因，神经元活动标志 | 高 |\n| **Npas4** | 神经元活动依赖的突触可塑性 | 中 |\n| **Bdnf** | 神经营养因子，学习记忆 | 中 |\n| **Srebf1** | 脂代谢 TF，睡眠剥夺后上调 | 中 |\n| **Homer1** | 突触后致密区蛋白 | 中 |\n| **Nr4a1/Nr4a2** | 孤核受体，应激反应 | 中 |\n\n建议从 GRN 分析中找出你数据里最活跃的 TF，然后做虚拟敲除验证。